# NPTEL Certificate verification project

### install all this libraries

In [ ]:
pip install pymupdf opencv-python numpy requests pyzbar



Note: you may need to restart the kernel to use updated packages.


### Import the libraries

In [18]:
import fitz  # PyMuPDF for PDF processing
import cv2
import numpy as np
import requests
import re
from pyzbar.pyzbar import decode

### Extracting the text from PDF certificate

In [26]:
pdf_path="certificate.pdf"

def extract_text_from_pdf(pdf_path):
    """Extracts all text from a given PDF file.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The extracted text from the PDF.
    """
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    return text

extract_text_from_pdf(pdf_path)



'No. of credits recommended: 3 or 4\nTo verify the certificate\nRoll No:\nJul-Oct 2024\n(12 week course)\nFundamentals of Artificial Intelligence\nSAMIR SRINATH\n19.69/25\n57.75/75\n77\n3260\nNPTEL24GE47S351801124\n\n'

### extract the roll no from the above text

In [25]:
def extract_roll_number(pdf_path):
    """Extracts the Roll Number from a certificate PDF.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The extracted Roll Number, or None if not found.
    """
    doc = fitz.open(pdf_path)
    text = doc[0].get_text("text")  # Extract text from the first page

    lines = text.split("\n")
    for line in reversed(lines):  # Reverse search for the last meaningful alphanumeric entry
        line = line.strip()
        if line.isalnum() and len(line) > 5:  # Assuming roll numbers are alphanumeric and sufficiently long
            print(f"✅ Extracted Roll No: {line}")
            return line

    print("❌ Roll number not found in certificate.")
    return None
extract_roll_number(pdf_path)


✅ Extracted Roll No: NPTEL24GE47S351801124


'NPTEL24GE47S351801124'

## Extracting the qr file from pdf

In [27]:
def extract_qr_from_pdf(pdf_path):
    """Extracts and decodes the QR Code from a certificate PDF.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The extracted QR Code URL, or None if no QR is found.
    """
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        images = doc[page_num].get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]  # Image reference
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to OpenCV image
            np_img = np.frombuffer(image_bytes, np.uint8)
            img_cv = cv2.imdecode(np_img, cv2.IMREAD_COLOR)

            # Try to decode QR Code
            qr_codes = decode(img_cv)
            for qr in qr_codes:
                qr_data = qr.data.decode("utf-8")
                print(f"✅ Extracted QR Code URL: {qr_data}")
                return qr_data  # Return first QR Code URL found

    print("❌ No QR Code found in the certificate.")
    return None  # No QR Code found
extract_qr_from_pdf(pdf_path)

✅ Extracted QR Code URL: https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251


'https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251'

## If the qr is valid or not

In [33]:
def is_valid_url(url):
    """Checks if a given URL is valid by sending an HTTP request.

    Args:
        url (str): The URL to check.

    Returns:
        bool: True if the URL is reachable (HTTP 200), otherwise False.
    """
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False
is_valid_url("https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251")

True

# **Total combined code**

In [35]:
import fitz  # PyMuPDF for PDF processing
import cv2
import numpy as np
import requests
import re
from pyzbar.pyzbar import decode

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    return text

def extract_roll_number(pdf_path):
    """Extracts the roll number by scanning the last alphanumeric string in the text."""
    doc = fitz.open(pdf_path)
    text = doc[0].get_text("text")  # Extract text from the first page
    
    lines = text.split("\n")
    for line in reversed(lines):  # Reverse search for the last meaningful alphanumeric entry
        line = line.strip()
        if line.isalnum() and len(line) > 5:  # Assuming roll numbers are alphanumeric and sufficiently long
            print(f"✅ Extracted Roll No: {line}")
            return line

    print("❌ Roll number not found in certificate.")
    return None

def extract_qr_from_pdf(pdf_path):
    """Extracts QR code from PDF and decodes its URL."""
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        images = doc[page_num].get_images(full=True)
        
        for img_index, img in enumerate(images):
            xref = img[0]  # Image reference
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to OpenCV image
            np_img = np.frombuffer(image_bytes, np.uint8)
            img_cv = cv2.imdecode(np_img, cv2.IMREAD_COLOR)

            # Try to decode QR Code
            qr_codes = decode(img_cv)
            for qr in qr_codes:
                qr_data = qr.data.decode("utf-8")
                print(f"✅ Extracted QR Code URL: {qr_data}")
                return qr_data  # Return first QR Code URL found

    print("❌ No QR Code found in the certificate.")
    return None  # No QR Code found

def is_valid_url(url):
    """Checks if a URL is reachable (returns HTTP 200)."""
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False

def verify_certificate(pdf_path):
    """Verifies the certificate by checking if Roll No is present in the QR URL and URL is valid."""
    # Extract roll number from the certificate text
    roll_no_from_text = extract_roll_number(pdf_path)

    # Extract QR code URL
    qr_url = extract_qr_from_pdf(pdf_path)

    # Check if URL is valid
    url_valid = is_valid_url(qr_url) if qr_url else False

    # Check if the roll number exists inside the QR URL (not exact match)
    roll_no_match = roll_no_from_text in qr_url if roll_no_from_text and qr_url else False

    # Final validation
    if roll_no_match and url_valid:
        print("✅ Certificate is VALID ✅")
        return True
    else:
        print("❌ Certificate is INVALID ❌")
        return False

# Run the verification
verify_certificate("certificate.pdf")


✅ Extracted Roll No: NPTEL24GE47S351801124
✅ Extracted QR Code URL: https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251
✅ Certificate is VALID ✅


True

### **For the testing of the multiple files**

In [42]:
import fitz  # PyMuPDF for PDF processing
import cv2
import numpy as np
import requests
import re
import os
from pyzbar.pyzbar import decode

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    return text

def extract_roll_number(pdf_path):
    """Extracts the roll number by scanning the last alphanumeric string in the text."""
    doc = fitz.open(pdf_path)
    text = doc[0].get_text("text")  # Extract text from the first page
    
    lines = text.split("\n")
    for line in reversed(lines):  # Reverse search for the last meaningful alphanumeric entry
        line = line.strip()
        if line.isalnum() and len(line) > 5:  # Assuming roll numbers are alphanumeric and sufficiently long
            print(f"✅ Extracted Roll No from {pdf_path}: {line}")
            return line

    print(f"❌ Roll number not found in {pdf_path}.")
    return None

def extract_qr_from_pdf(pdf_path):
    """Extracts QR code from PDF and decodes its URL."""
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        images = doc[page_num].get_images(full=True)
        
        for img_index, img in enumerate(images):
            xref = img[0]  # Image reference
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to OpenCV image
            np_img = np.frombuffer(image_bytes, np.uint8)
            img_cv = cv2.imdecode(np_img, cv2.IMREAD_COLOR)

            # Try to decode QR Code
            qr_codes = decode(img_cv)
            for qr in qr_codes:
                qr_data = qr.data.decode("utf-8")
                print(f"✅ Extracted QR Code URL from {pdf_path}: {qr_data}")
                return qr_data  # Return first QR Code URL found

    print(f"❌ No QR Code found in {pdf_path}.")
    return None  # No QR Code found

def is_valid_url(url):
    """Checks if a URL is reachable (returns HTTP 200)."""
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False

def verify_certificate(pdf_path):
    """Verifies the certificate by checking if Roll No is present in the QR URL and URL is valid."""
    # Extract roll number from the certificate text
    roll_no_from_text = extract_roll_number(pdf_path)

    # Extract QR code URL
    qr_url = extract_qr_from_pdf(pdf_path)

    # Check if URL is valid
    url_valid = is_valid_url(qr_url) if qr_url else False

    # Check if the roll number exists inside the QR URL (not exact match)
    roll_no_match = roll_no_from_text in qr_url if roll_no_from_text and qr_url else False

    # Final validation
    if roll_no_match and url_valid:
        print(f"✅ Certificate {pdf_path} is VALID ✅\n")
        return True
    else:
        print(f"❌ Certificate {pdf_path} is INVALID ❌\n")
        return False

def verify_all_certificates(folder_path):
    """Verifies all certificate PDFs in a given folder."""
    print("\n🔍 Scanning folder for certificates...\n")
    
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith(".pdf")]
    
    if not pdf_files:
        print("⚠️ No PDF files found in the folder.")
        return

    for pdf in pdf_files:
        pdf_path = os.path.join(folder_path, pdf)
        print(f"🔹 Processing: {pdf}")
        verify_certificate(pdf_path)

# Run verification for all PDFs in the "certificates" folder
verify_all_certificates("certificates")



🔍 Scanning folder for certificates...

🔹 Processing: certificate.pdf
✅ Extracted Roll No from certificates\certificate.pdf: NPTEL24GE47S351801124
✅ Extracted QR Code URL from certificates\certificate.pdf: https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251
✅ Certificate certificates\certificate.pdf is VALID ✅

🔹 Processing: test2.pdf
❌ Roll number not found in certificates\test2.pdf.
✅ Extracted QR Code URL from certificates\test2.pdf: https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251
❌ Certificate certificates\test2.pdf is INVALID ❌

🔹 Processing: test3.pdf
✅ Extracted Roll No from certificates\test3.pdf: NPTEL24GE47S351801124
✅ Extracted QR Code URL from certificates\test3.pdf: https://nptel.ac.in/noc/E_Certificate/NPTEL24GE47S35180112404265251
✅ Certificate certificates\test3.pdf is VALID ✅

🔹 Processing: test4.pdf
✅ Extracted Roll No from certificates\test4.pdf: NPTEL24GE47S351801124
✅ Extracted QR Code URL from certificates\test4.pdf: https://npt